# Phase 1: Robustness to Class Imbalance (RQ4)

**Objective**: Determine if White-Box and Black-Box representations degrade when trained on highly skewed underlying label distributions. We enforce a strict control baseline to guarantee metric differences are driven exclusively by distribution skew, not sample volume.


# Colab & Environment Setup

In [ ]:
try:
    import google.colab
    IN_COLAB = True
    print("Running in Google Colab. Installing dependencies...")
    get_ipython().system('pip install -q transformers accelerate datasets scikit-learn pandas')
    
    from google.colab import drive
    drive.mount('/content/drive')
    
    import sys, os
    
    # Path to new Class Balance Folder
    base_proj_path = '/content/drive/MyDrive/llm_lie_detection_black_vs_white_box/Black_vs_White_Lie_Detection/Class_Balance_Impact_BB_WB'
    os.chdir(base_proj_path)
    
    # Add Aligned_Comparison_BB_WB to sys path so we can import unmodified core utils
    aligned_path = os.path.abspath(os.path.join(base_proj_path, '..', 'Aligned_Comparison_BB_WB'))
    sys.path.insert(0, aligned_path)
    sys.path.insert(0, base_proj_path)
    print(f"✅ Executing in: {os.getcwd()}")
    
    from huggingface_hub import login
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token: login(token=hf_token)
        
except ImportError:
    IN_COLAB = False
    import sys, os
    # Local fallback
    current_dir = os.getcwd()
    aligned_path = os.path.abspath(os.path.join(current_dir, '..', 'Aligned_Comparison_BB_WB'))
    sys.path.insert(0, aligned_path)
    sys.path.insert(0, current_dir)

# Imports & Configuration

In [ ]:
import torch
import numpy as np
import pandas as pd
import time
from tqdm.auto import tqdm

# Import unmodified feature extractors from Aligned Comparison
from wb_activations import load_model, get_activations_for_dataset
from bb_logprobs import compute_bb_features_for_dataset

# Import customized Imbalance Evaluation Code
from load_data_imbalance import load_dataset_imbalance
from wb_probes_imbalance import train_wb_probes_imbalance
from bb_classifier_imbalance import train_bb_classifier_imbalance
from evaluation_utils_imbalance import compare_results_imbalance

MODEL_NAME = "meta-llama/Llama-2-7b-chat-hf"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")


# 1. Create Datasets (Scenario A vs Scenario B)

In [ ]:
# We don't need DATASETS_CONFIG ratios anymore because the scenario flag handles exact question-grouping logic natively.
DATASETS_CONFIG = ["commonsense_qa", "arc_challenge", "open_book_qa"]

dataset_dict = {}

N_SAMPLES = 1000 

print("Curating highly controlled class distributions by question ID...\n")
for ds_name in DATASETS_CONFIG:
    print(f"--- Processing {ds_name} ---")
    
    # Setup A: Balanced (Strictly 1 Truth + 1 Lie per Question)
    df_train_A = load_dataset_imbalance(ds_name, split='train', n_samples=N_SAMPLES, scenario="A")
    
    # Setup B: Imbalanced (Strictly 1 Truth + ALL Lies per Question)
    df_train_B = load_dataset_imbalance(ds_name, split='train', n_samples=N_SAMPLES, scenario="B")
    
    # Validation: Strictly controlled Balanced (Scenario A logic) to act as the perfect ground-truth exam
    df_val = load_dataset_imbalance(ds_name, split='validation', n_samples=min(N_SAMPLES, 400), scenario="A")
    
    if len(df_train_A) > 0 and len(df_train_B) > 0:
        dataset_dict[ds_name] = {
            "train_A": df_train_A,
            "train_B": df_train_B,
            "validation": df_val
        }
        print(f"✅ {ds_name} Loaded -> Setup A (Balanced): {len(df_train_A)} | Setup B (Imbalanced): {len(df_train_B)} | Val: {len(df_val)}\n")

# Make DATASETS list available for the remaining cells
DATASETS = list(dataset_dict.keys())

# 2. Load the model

In [ ]:
print("Loading Model...")
model, tokenizer = load_model(MODEL_NAME, device=DEVICE)

# 3. Extract activations vectors for white box

In [ ]:
import gc

WB_BATCH_SIZE = 64 # White Box needs all hidden states, limiting memory 

wb_features = {}
for ds_name in tqdm(DATASETS, desc="Extracting Model Geography"):
    # Dynamically lower batch size for memory-heavy datasets (like IMDB if added later)
    current_batch_size = 28 if ds_name == "imdb" else WB_BATCH_SIZE

    # 1. Extract White Box Activations
    act_train_A = get_activations_for_dataset(dataset_dict[ds_name]["train_A"], model, tokenizer, device=DEVICE, batch_size=current_batch_size)
    act_train_B = get_activations_for_dataset(dataset_dict[ds_name]["train_B"], model, tokenizer, device=DEVICE, batch_size=current_batch_size)
    act_val = get_activations_for_dataset(dataset_dict[ds_name]["validation"], model, tokenizer, device=DEVICE, batch_size=current_batch_size)
    
    wb_features[ds_name] = {"train_A": act_train_A, "train_B": act_train_B, "validation": act_val}
    print(f"✅ {ds_name} White Box Features Extracted")
    
    # 🧹 Thoroughly clean GPU and RAM memory after each dataset
    torch.cuda.empty_cache()
    gc.collect()


# 4. Compute logprobs for black box

In [ ]:
import os
import gc

# Load the exact elicitation questions used in the original Black Box paper
try:
    # We navigate from Class_Balance_Impact_BB_WB to the exact folder
    probe_base = "../../Black_Box_Lie_Detection/results/probes_groups"
    no_lie_indices = np.load(os.path.join(probe_base, "no_lie_indices.npy"))
    lie_indices = np.load(os.path.join(probe_base, "lie_indices.npy"))
    knowable_indices = np.load(os.path.join(probe_base, "knowable_indices.npy"))
    PAPER_PROBE_INDICES = np.concatenate([no_lie_indices, lie_indices, knowable_indices])
    print(f"✅ Loaded {len(PAPER_PROBE_INDICES)} elicitation questions (exact same as original paper)")
except FileNotFoundError as e:
    print(f"⚠️ Warning: Could not load original paper's probe indices: {e}")
    print("Will use all available probes instead.")
    PAPER_PROBE_INDICES = None

BB_BATCH_SIZE = 192 # Black Box only needs logits, so we can stack much larger batches

bb_features = {}
for ds_name in tqdm(DATASETS, desc="Extracting Model Geography"):
    # 2. Compute Black Box Logprobs
    bb_train_A = compute_bb_features_for_dataset(
        dataset_dict[ds_name]["train_A"], model, tokenizer, device=DEVICE, batch_size=BB_BATCH_SIZE, probe_indices=PAPER_PROBE_INDICES
    )
    bb_train_B = compute_bb_features_for_dataset(
        dataset_dict[ds_name]["train_B"], model, tokenizer, device=DEVICE, batch_size=BB_BATCH_SIZE, probe_indices=PAPER_PROBE_INDICES
    )
    bb_val = compute_bb_features_for_dataset(
        dataset_dict[ds_name]["validation"], model, tokenizer, device=DEVICE, batch_size=BB_BATCH_SIZE, probe_indices=PAPER_PROBE_INDICES
    )
    
    # Unpack BB to numpy matrices aligned precisely
    def unpack_bb(bb_dict):
        X, Y = [], []
        for r in bb_dict:
            if "bb_features" in r and r["bb_features"]:
                X.append([p["score"] for p in r["bb_features"]])
                Y.append(r["label"])
        return np.array(X), np.array(Y)
    
    bb_features[ds_name] = {
        "train_A": unpack_bb(bb_train_A),
        "train_B": unpack_bb(bb_train_B),
        "validation": unpack_bb(bb_val)
    }
    
    # 🧹 Thoroughly clean GPU and RAM memory after each dataset
    torch.cuda.empty_cache()
    gc.collect()

# 5. Train Classifiers and Expose Imbalance Impact

In [ ]:
# 1. Expand Layer List from 1st layer to 32nd layer (Step of 2)
LAYER_LIST = list(range(1, 33, 2)) 

# 2. Include all 8 White-Box probe variations available in wb_probes.py
WB_METHODS = ["lr", "dim", "lat", "ccs", "lda", "pca", "pca-g", "lr-g"]

all_results = []

for ds_name in DATASETS:
    print(f"\n--- Training Setups for {ds_name} ---")
    
    # Data Pointers
    w_tr_A, w_tr_B, w_v = wb_features[ds_name]["train_A"], wb_features[ds_name]["train_B"], wb_features[ds_name]["validation"]
    b_tr_A, b_tr_B, b_v = bb_features[ds_name]["train_A"], bb_features[ds_name]["train_B"], bb_features[ds_name]["validation"]
    X_val_bb, y_val_bb = b_v
    
    # ----- SCENARIO A: BALANCED -----
    print(f">>> Executing Setup A (Balanced 50/50) for {ds_name}")
    bb_model_A, bb_met_A = train_bb_classifier_imbalance(b_tr_A[0], X_val_bb, b_tr_A[1], y_val_bb)
    res_A_matrices = []
    
    for method in WB_METHODS:
        print(f"Training WB Probe: {method.upper()} on Scenario A...")
        _, wb_met_A = train_wb_probes_imbalance(w_tr_A, eval_data=w_v, layer_list=LAYER_LIST, method=method)
        df_A = compare_results_imbalance(wb_met_A, bb_met_A)
        df_A.insert(0, "Condition", "Setup A (Balanced)")
        df_A.insert(0, "Algorithm", method.upper())
        df_A.insert(0, "Dataset", ds_name)
        res_A_matrices.append(df_A)
        
    # ----- SCENARIO B: IMBALANCED -----
    print(f"\n>>> Executing Setup B (Imbalanced) for {ds_name}")
    bb_model_B, bb_met_B = train_bb_classifier_imbalance(b_tr_B[0], X_val_bb, b_tr_B[1], y_val_bb)
    res_B_matrices = []
    
    for method in WB_METHODS:
        print(f"Training WB Probe: {method.upper()} on Scenario B...")
        _, wb_met_B = train_wb_probes_imbalance(w_tr_B, eval_data=w_v, layer_list=LAYER_LIST, method=method)
        df_B = compare_results_imbalance(wb_met_B, bb_met_B)
        df_B.insert(0, "Condition", "Setup B (Imbalanced)")
        df_B.insert(0, "Algorithm", method.upper())
        df_B.insert(0, "Dataset", ds_name)
        res_B_matrices.append(df_B)
        
    all_results.extend(res_A_matrices)
    all_results.extend(res_B_matrices)

final_df = pd.concat(all_results, ignore_index=True)


# 6. View Imbalance Degradation Metrics


In [ ]:
# Save raw data to disk
final_df.to_csv("class_imbalance_metrics_detailed.csv", index=False)

# Cleanly separate Scenario A from B focusing on peak performance to show visual difference
from IPython.display import display

# Group by the features and aggregate max AUC/F1
summary_df = final_df.groupby(["Dataset", "Condition", "Algorithm"]).agg({
    "Macro F1": "max",
    "Recall (True)": "max",
    "Recall (False)": "max",
    "AUC": "max",
}).reset_index()

# Pivot the table so we can see Balanced vs Imbalanced side by side per dataset and algorithm
pivot_df = summary_df.pivot_table(
    index=["Dataset", "Algorithm"], 
    columns="Condition", 
    values=["Macro F1", "Recall (True)", "AUC"]
)

# Format the output beautifully
print("🌟 PHASE 1 EXPERIMENT COMPLETE: CLASS IMBALANCE RESULTS 🌟")
print("Observe how the natural Imbalanced setup degrades Recall (True) and Macro F1 compared to the Balanced setup, ")
print("showing that unbalanced datasets inherently train models to just guess 'False'.\n")

display(pivot_df)

# Optionally, save this pivoted highly readable table to a CSV for your thesis
pivot_df.to_csv("class_imbalance_summary_table.csv")